In [ ]:
!pip install transformers sentence-transformers torch pdfplumber scikit-learn spacy python-dotenv
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 36.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 58.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import os

# Define las rutas de las carpetas
PDF_DIR = "data/pdfs"
SUMMARY_DIR = "outputs/summaries"
CLUSTER_DIR = "outputs/clusters"

# Crea las carpetas si no existen
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)
os.makedirs(CLUSTER_DIR, exist_ok=True)

print(f"Directorios creados o ya existen: {PDF_DIR}, {SUMMARY_DIR}, {CLUSTER_DIR}")

Directorios creados o ya existen: data/pdfs, outputs/summaries, outputs/clusters


In [ ]:
import os
import re
import json
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import pdfplumber
import spacy
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pdfminer.pdfpage import PDFPage
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfdocument import PDFDocument
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.converter import TextConverter
from pdfminer.layout import LAParams
from io import StringIO

# ---------------------------
# Configuración y utilidades
# ---------------------------

@dataclass
class YachaiConfig:
    pdf_dir: str = "data/pdfs"
    out_summary_dir: str = "outputs/summaries"
    out_cluster_dir: str = "outputs/clusters"
    # Modelo de resumen: BART en inglés o T5 multilingüe; si tu biblio está en español, T5 puede funcionar mejor
    summarizer_model: str = "t5-small"  # Cambiado a un modelo T5, que es multilingüe y puede manejar español
    embedding_model: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    language_model_spacy: str = "es_core_news_sm"
    max_pdf_pages: int = 50     # límite para prototipo (evita PDFs muy largos)
    chunk_size_chars: int = 1500  # dividir textos largos en chunks para resumir (ajustado para t5-small)

def ensure_dirs(cfg: YachaiConfig):
    os.makedirs(cfg.out_summary_dir, exist_ok=True)
    os.makedirs(cfg.out_cluster_dir, exist_ok=True)
    # Asegurarse de que el directorio de PDFs también se cree
    os.makedirs(cfg.pdf_dir, exist_ok=True)

# ---------------------------
# Capa de lectura y limpieza
# ---------------------------

class DocumentLoader:
    def __init__(self, cfg: YachaiConfig):
        self.cfg = cfg

    def load_pdfs(self) -> Dict[str, str]:
        texts = {}
        # Asegurarse de que el directorio existe antes de listar
        if not os.path.exists(self.cfg.pdf_dir):
            print(f"[Yacha.i.] Advertencia: El directorio de PDFs '{self.cfg.pdf_dir}' no existe. Creando...")
            os.makedirs(self.cfg.pdf_dir, exist_ok=True)

        for fname in os.listdir(self.cfg.pdf_dir):
            if not fname.lower().endswith(".pdf"): # Ignorar archivos que no son PDF
                print(f"[Yacha.i.] Ignorando archivo no PDF: {fname}")
                continue
            path = os.path.join(self.cfg.pdf_dir, fname)
            texts[fname] = self._extract_text_pdf(path)
        return texts

    def _extract_text_pdf(self, path: str) -> str:
        pages_text = []
        try:
            with pdfplumber.open(path) as pdf:
                for i, page in enumerate(pdf.pages):
                    if i >= self.cfg.max_pdf_pages:
                        break
                    text = page.extract_text() or ""
                    pages_text.append(text)
            raw = "\n".join(pages_text)
            return self._clean_text(raw)
        except pdfplumber.pdf.PdfminerException as e:
            print(f"[Yacha.i.] Error al procesar PDF '{path}': {e}. Este archivo será omitido.")
            return ""
        except Exception as e:
            print(f"[Yacha.i.] Un error inesperado ocurrió al procesar PDF '{path}': {e}. Este archivo será omitido.")
            return ""

    @staticmethod
    def _clean_text(text: str) -> str:
        text = re.sub(r"\s+", " ", text)     # espacios múltiples -> uno
        text = re.sub(r"[^\S\r\n]+", " ", text)
        text = text.strip()
        return text

# ---------------------------
# NLP: resumen y conceptos
# ---------------------------

class Summarizer:
    def __init__(self, cfg: YachaiConfig):
        # Para español extensivo, considera "mrm8488/bert2bert_shared-spanish-finetuned-summarization"
        self.summarizer = pipeline("summarization", model=cfg.summarizer_model)
        self.cfg = cfg
        self.nlp = spacy.load(cfg.language_model_spacy)

    def summarize_long(self, text: str, title: str) -> Dict[str, Any]:
        if not text.strip():  # Si el texto está vacío, no intentar resumir
            print(f"[Yacha.i.] Advertencia: El documento '{title}' está vacío o no se pudo extraer texto. No se generará resumen.")
            return {
                "title": title,
                "chunks": [],
                "executive_summary": "No se pudo generar resumen por falta de contenido.",
                "key_concepts": []
            }

        chunks = self._chunk(text, self.cfg.chunk_size_chars)
        chunk_summaries = []
        for chunk in chunks:
            # Usar max_new_tokens para controlar la longitud de la salida
            # min_new_tokens asegura una longitud mínima.
            # do_sample=False es para generar resúmenes deterministas.
            summary_output = self.summarizer(chunk, max_new_tokens=220, min_new_tokens=80, do_sample=False)
            if summary_output and summary_output[0] and "summary_text" in summary_output[0]:
                chunk_summaries.append(summary_output[0]["summary_text"])
            else:
                print(f"[Yacha.i.] Advertencia: El resumen de un chunk del documento '{title}' falló o no produjo texto.")
                chunk_summaries.append("") # Añadir cadena vacía para no romper el " ".join

        # unificar resúmenes de chunks
        unified = " ".join([s for s in chunk_summaries if s.strip()]) # Unir solo resúmenes no vacíos

        # resumen ejecutivo final
        final_summary_output = self.summarizer(unified, max_new_tokens=200, min_new_tokens=80, do_sample=False)
        if final_summary_output and final_summary_output[0] and "summary_text" in final_summary_output[0]:
            final = final_summary_output[0]["summary_text"]
        else:
            print(f"[Yacha.i.] Advertencia: El resumen ejecutivo final del documento '{title}' falló o no produjo texto.")
            final = "No se pudo generar resumen ejecutivo."

        # extracción de conceptos clave (sencilla: sustantivos y entidades)
        concepts = self._extract_key_concepts(unified)

        return {
            "title": title,
            "chunks": chunk_summaries,
            "executive_summary": final,
            "key_concepts": concepts
        }

    def _chunk(self, text: str, size: int) -> List[str]:
        return [text[i:i+size] for i in range(0, len(text), size)] if text else []

    def _extract_key_concepts(self, text: str, top_n: int = 15) -> List[str]:
        doc = self.nlp(text)
        candidates = []
        # Sustantivos y frases nominales simples
        candidates += [t.lemma_.lower() for t in doc if t.pos_ in ("NOUN", "PROPN") and len(t.text) > 2]
        # Entidades nombradas
        candidates += [ent.text.lower() for ent in doc.ents if len(ent.text) > 2]
        # Conteo simple
        freq: Dict[str, int] = {}
        for c in candidates:
            freq[c] = freq.get(c, 0) + 1
        # Top-N
        sorted_items = sorted(freq.items(), key=lambda x: x[1], reverse=True)
        # Filtrar ruido básico
        stop = set(["figura", "tabla", "introducción", "conclusión", "resumen", "artículo", "estudio"])
        concepts = [w for w, n in sorted_items if w not in stop][:top_n]
        return concepts

# ---------------------------
# Embeddings y clustering
# ---------------------------

class ThematicOrganizer:
    def __init__(self, cfg: YachaiConfig):
        self.embedder = SentenceTransformer(cfg.embedding_model)
        self.cfg = cfg

    def build_corpus_embeddings(self, summaries: Dict[str, Dict[str, Any]]) -> Tuple[List[List[float]], List[str]]:
        texts = []
        names = []
        for fname, data in summaries.items():
            # Asegurarse de que el resumen ejecutivo no esté vacío
            if data["executive_summary"] and data["executive_summary"] != "No se pudo generar resumen por falta de contenido.":
                # usar resumen ejecutivo + conceptos para representación compacta
                rep = data["executive_summary"] + " | " + ", ".join(data["key_concepts"])
                texts.append(rep)
                names.append(fname)
            else:
                print(f"[Yacha.i.] Advertencia: El documento '{fname}' no tiene resumen válido para crear embedding y será omitido del clustering.")

        if not texts:
            print("[Yacha.i.] No hay textos válidos para generar embeddings y realizar clustering.")
            return [], []

        embeddings = self.embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
        return embeddings, names

    def cluster(self, embeddings, names, k: int = None) -> Dict[str, Any]:
        if not embeddings or len(embeddings) < 2: # Necesitamos al menos 2 documentos para clusterizar
            print("[Yacha.i.] No hay suficientes documentos para realizar clustering (se necesitan al menos 2).")
            return {"k": 0, "clusters": {}}

        # Si no se especifica k, tratar de estimar un rango razonable
        if k is None:
            # Ajustar k_candidates para que no exceda el número de embeddings disponibles
            max_k = min(10, len(embeddings))
            if max_k < 2:
                print("[Yacha.i.] No hay suficientes documentos para estimar un número de clusters con Silhouette Score.")
                return {"k": 1, "clusters": {0: names}} # Si solo hay 1 documento, ponlo en el cluster 0

            k_candidates = list(range(2, max_k + 1))

            if not k_candidates: # Si solo hay un documento o no hay suficientes para k=2
                return {"k": 1, "clusters": {0: names}}

            best_k = 2
            best_score = -1
            for kc in k_candidates:
                km = KMeans(n_clusters=kc, n_init=10, random_state=42)
                labels = km.fit_predict(embeddings)
                # Solo calcular silhouette_score si hay más de un cluster posible
                if len(set(labels)) > 1:
                    score = silhouette_score(embeddings, labels)
                    if score > best_score:
                        best_score = score
                        best_k = kc
                else:
                    # Si solo se forma un cluster, el score no es significativo o no se puede calcular
                    if kc == 1: # Si solo hay un cluster, lo consideramos como 'best' si no hay otra opción
                        best_k = 1
                        best_score = 0.0
            k = best_k

        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(embeddings)

        clusters: Dict[int, List[str]] = {}
        for name, lab in zip(names, labels):
            clusters.setdefault(lab, []).append(name)

        return {
            "k": k,
            "clusters": clusters
        }

# ---------------------------
# Orquestador Yacha.i.
# ---------------------------

class YachaiAssistant:
    def __init__(self, cfg: YachaiConfig):
        self.cfg = cfg
        ensure_dirs(cfg)
        self.loader = DocumentLoader(cfg)
        self.summarizer = Summarizer(cfg)
        self.organizer = ThematicOrganizer(cfg)

    def run(self) -> Dict[str, Any]:
        # 1) Asegurarse de que el directorio de PDFs exista antes de leer
        ensure_dirs(self.cfg)
        print(f"[Yacha.i.] Verificando y creando directorios. PDF_DIR: {self.cfg.pdf_dir}")

        # 2) Leer PDFs
        pdf_texts = self.loader.load_pdfs()
        if not pdf_texts:
            print("No se encontraron PDFs válidos o procesables en:", self.cfg.pdf_dir)
            return {}

        # 3) Resumir y extraer conceptos
        summaries: Dict[str, Dict[str, Any]] = {}
        for fname, text in pdf_texts.items():
            if text.strip(): # Solo intentar resumir si hay texto extraído
                print(f"[Yacha.i.] Resumiendo: {fname}")
                s = self.summarizer.summarize_long(text, title=fname)
                summaries[fname] = s
                print(f"[DEBUG] Resumen ejecutivo para {fname}: {s.get('executive_summary', 'N/A')[:100]}...") # Added debug print for s
                # Guardar cada resumen
                out_path = os.path.join(self.cfg.out_summary_dir, fname.replace(".pdf", "_summary.json"))
                with open(out_path, "w", encoding="utf-8") as f:
                    json.dump(s, f, ensure_ascii=False, indent=2)
            else:
                print(f"[Yacha.i.] El documento '{fname}' está vacío o no se pudo extraer texto. Se omitirá el resumen.")

        if not summaries:
            print("[Yacha.i.] No se generaron resúmenes para ningún documento. Fin del proceso.")
            return {}
        print(f"[DEBUG] Claves del diccionario 'summaries' después del bucle de resumen: {list(summaries.keys())}") # Added debug

        # 4) Embeddings y clustering temático
        embeddings, names = self.organizer.build_corpus_embeddings(summaries)
        if not embeddings:
            # Si no hay embeddings, aún devolvemos los resúmenes generados.
            print("[Yacha.i.] No se pudieron generar embeddings o no hay suficientes documentos para el clustering.")
            return {"summaries": summaries, "clusters": {"k": 0, "clusters": {}}}

        clustering = self.organizer.cluster(embeddings, names, k=None)
        # Guardar clusters
        out_cluster = os.path.join(self.cfg.out_cluster_dir, "clusters.json")
        with open(out_cluster, "w", encoding="utf-8") as f:
            json.dump(clustering, f, ensure_ascii=False, indent=2)

        print(f"[Yacha.i.] Resúmenes guardados en {self.cfg.out_summary_dir}")
        print(f"[Yacha.i.] Clusters guardados en {self.cfg.out_cluster_dir}")
        return {
            "summaries": summaries,
            "clusters": clustering
        }

# ---------------------------
# Ejecución
# ---------------------------

if __name__ == "__main__":
    cfg = YachaiConfig()
    yachai = YachaiAssistant(cfg)
    result = yachai.run()
    # Vista rápida en consola
    if result and "clusters" in result and result["clusters"] and result["clusters"]["clusters"]:
        print("\n=== Vista rápida de clusters (Yacha.i.) ===")
        for cid, files in result["clusters"]["clusters"].items():
            print(f"Cluster {cid}:")
            for f in files:
                print(f"  - {f}")
    elif result: # Si hay resúmenes pero no clusters válidos
        print("\n=== Vista rápida (Yacha.i.) ===")
        print("Se procesaron documentos y se generaron resúmenes, pero no se pudo realizar clustering temático.\nVerifica el directorio de resúmenes para ver los archivos generados.")
    else:
        print("\n=== Vista rápida (Yacha.i.) ===")
        print("No se pudo procesar ningún documento o generar resultados. Verifica los mensajes anteriores para más detalles.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Yacha.i.] Verificando y creando directorios. PDF_DIR: data/pdfs
No se encontraron PDFs válidos o procesables en: data/pdfs

=== Vista rápida (Yacha.i.) ===
No se pudo procesar ningún documento o generar resultados. Verifica los mensajes anteriores para más detalles.


Primero, crearemos un archivo JSON de ejemplo para trabajar con él:

In [ ]:
import json

data = {
    "nombre": "Juan",
    "edad": 30,
    "ciudad": "Madrid",
    "intereses": ["programacion", "lectura", "senderismo"]
}

file_path = "ejemplo.json"
with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"Archivo '{file_path}' creado con éxito.")

Archivo 'ejemplo.json' creado con éxito.


Ahora, para leer el archivo JSON, puedes hacer lo siguiente:

In [ ]:
import json

file_path = "ejemplo.json"

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        loaded_data = json.load(f)
    print(f"Datos cargados de '{file_path}':")
    print(loaded_data)
    print(f"Tipo de datos cargados: {type(loaded_data)}")
except FileNotFoundError:
    print(f"Error: El archivo '{file_path}' no se encontró.")
except json.JSONDecodeError:
    print(f"Error: El archivo '{file_path}' no es un JSON válido.")

Datos cargados de 'ejemplo.json':
{'nombre': 'Juan', 'edad': 30, 'ciudad': 'Madrid', 'intereses': ['programacion', 'lectura', 'senderismo']}
Tipo de datos cargados: <class 'dict'>


In [ ]:
import os

pdf_directory = "data/pdfs"

print(f"Contenido de {pdf_directory}:")
if os.path.exists(pdf_directory) and os.listdir(pdf_directory):
    for filename in os.listdir(pdf_directory):
        print(f"- {filename}")
else:
    print(f"El directorio {pdf_directory} está vacío o no existe.")

Contenido de data/pdfs:
El directorio data/pdfs está vacío o no existe.


In [ ]:
import os

pdf_directory = "data/pdfs"

print(f"Contenido de {pdf_directory}:")
if os.path.exists(pdf_directory) and os.listdir(pdf_directory):
    for filename in os.listdir(pdf_directory):
        print(f"- {filename}")
else:
    print(f"El directorio {pdf_directory} está vacío o no existe.")

Contenido de data/pdfs:
El directorio data/pdfs está vacío o no existe.
